# Environment Setup

This notebook is designed for Google Colab.

Before running, ensure:
- Google Drive is mounted.
- The dataset zip exists at `/content/drive/MyDrive/datasets/celeba.zip`.
- You run cells top-to-bottom at least once to initialize all variables.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Create the directory if it doesn't exist
!mkdir -p /content/datasets

In [ ]:
# This should take 1-2 minutes
# It unzips the dataset in the runtime's local SSD, so when
# you disconnect, it gets deleted
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

### Imports and Utilities

In [ ]:
import torch
from pathlib import Path
import os
import matplotlib.pyplot as plt
import numpy as np

from transformers import CLIPModel, CLIPProcessor
from torchvision.datasets import CelebA
import torchvision

In [ ]:
def plot_images(celeba_dataset, indices, n_cols, n_rows, figsize=(20, 10)):
    """Helper function to plot group of images"""
    if len(indices) > n_cols * n_rows:
        raise ValueError("Number of indices exceeds the grid capacity")
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)

    for counter, img_idx in enumerate(indices):
        img, _ = celeba_dataset[img_idx]
        ax = axes[counter // n_cols, counter % n_cols]
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# Data loading and Exploration
In this section, we load the dataset and perform an initial exploration to understand its structure and main characteristics.

The dataset used is `CelebA`, which contains 19,962 samples. Each sample consists of:
- an image of size `178 × 218`, representing a face of a celebrity;
- a set of 40 attributes that describe visual features of the person in the image.

We will briefly inspect the dataset by visualizing some samples and examining the associated attributes, in order to get a better understanding of the data before proceeding with further analysis.

In [ ]:
# Do *not* put `celeba` in the path.
# The dataset class will do that automatically!
data_root = Path("/content/datasets")
celeba = CelebA(root=data_root, split="test", download=False)

# This should be 19.962
print("Number of samples:", len(celeba))

# show element size
sample_img, sample_attrs = celeba[0]
print(f"Sample image size: {sample_img.size}")
print(f"Number of attributes: {len(sample_attrs)}")

### Sample visualization
First, we can visualize a random selection of images from the dataset to get a sense of the variety and quality of the images. We will display 50 random images in a grid format.

In [ ]:
# Get 50 random images and visualize them
indices = np.random.choice(len(celeba), size=50, replace=False)
plot_images(celeba, indices=indices, n_cols=10, n_rows=5)

### Attribute annotation
Now that we know how to load our dataset and we have visualized some samples, let's move to understanding how attributes are annotated in the dataset. Each image in the dataset is annotated with a set of 40 binary attributes, from the following list. 

Here, we also report how frequently each attribute appears in the dataset, which is important to understand the distribution of attributes and to design a retrieval system that can handle rare attributes effectively.


In [ ]:
all_labels = np.array([labels for _, labels in celeba])

attr_counts = all_labels.sum(axis=0)
attr_freq = all_labels.mean(axis=0)

print(f"{'Attribute':<20} {'Count':>10} {'Frequency':>10}")
print("-" * 45)

for attr, count, freq in zip(celeba.attr_names, attr_counts, attr_freq):
    print(f"{attr:<20} {count:>10} {freq:>10.3f}")

Let's define few other utilities functions that will facilitate the handling of attributes later on.
We will create a:
- `inx2attribute`: mapping from indices to attributes (e.g., index 0 corresponds to "5_o_Clock_Shadow", index 1 corresponds to "Arched_Eyebrows", etc.)

- `attribute2index`: mapping from attributes to indices (e.g., "5_o_Clock_Shadow" corresponds to index 0, "Arched_Eyebrows" corresponds to index 1, etc.)

- `retrieve_by_attributes(parameters)`: function that retrieves images based on specified attributes. This function will be crucial for our image retrieval system, allowing us to query the dataset for images that match certain attribute criteria.

- `plot_image_with_attributes`: function that displays an image along with its active attributes.

Query format for `retrieve_by_attributes`:
- Use `"+"` when the attribute must be present.
- Use `"-"` when the attribute must be absent.

Example: `{"Bald": "+", "Eyeglasses": "-"}`.

In [ ]:
# Assign a unique index to each attribute, and get the inverse mapping
idx2attribute = {idx: name for idx, name in enumerate(celeba.attr_names)}
attribute2idx = {name: idx for idx, name in enumerate(celeba.attr_names)}

def retrieve_by_attributes(parameters:dict):
    """
    Helper function that retrieve all the images that satisfy the conditions specified 
    in `parameters`.
    Params:
        - parameters: a dictionary where keys are attribute names and values are either "+" (must have the attribute) or "-" (must not have the attribute).  
    Returns:
        - A list of indices of images that satisfy the specified conditions.
    """
    # Start with all indices
    valid_indices = set(range(len(celeba)))
    
    # For each attribute condition, filter the indices
    for attr_name, value in parameters.items():
        attr_idx = attribute2idx[attr_name]
        if value == "+":
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 0:
                    valid_indices.remove(idx)
        elif value == "-":
            # Must not have the attribute
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 1:
                    valid_indices.remove(idx)
        else:
            raise ValueError(f"Invalid value for attribute condition: {value}. Use '+' or '-'.")
    
    return list(valid_indices)

def plot_image_with_attributes(idx):
    img, labels = celeba[idx]
    active_attrs = [idx2attribute[idx] for idx, value in enumerate(labels) if value == 1]

    fig, (ax_img, ax_text) = plt.subplots(1, 2, figsize=(10, 5))
    # Left: image
    ax_img.imshow(img)
    ax_img.axis('off')

    # Right: centered text
    ax_text.axis('off')
    text = "\n".join(active_attrs)

    ax_text.text(
        0.5, 0.5, text,
        fontsize=10,
        ha='center',   # horizontal alignment
        va='center'    # vertical alignment
    )

    plt.tight_layout()
    plt.show()

Now that we have the mapping, we can easily get the attributes of any image in the dataset. For example, let's get the attributes of a given image index.

In [ ]:
IMAGE_INDEX = 99
plot_image_with_attributes(99)

Now that we have everything in place, let's try to analyze some possible queries.


In [ ]:
query_1 = {"Bald": "+",
           "Smiling": "+",
           "Eyeglasses": "-",
           }
retrieved_images = retrieve_by_attributes(query_1)
print(f"Number of retrieved images: {len(retrieved_images)}")

# Plot up to 10 random retrieved images (without replacement).
n_samples = min(10, len(retrieved_images))
if n_samples == 0:
    print("No images match this query.")
else:
    sampled_indices = np.random.choice(retrieved_images, size=n_samples, replace=False)
    n_cols = 5
    n_rows = int(np.ceil(n_samples / n_cols))
    plot_images(celeba, indices=sampled_indices, n_cols=n_cols, n_rows=n_rows)

# Offline Feature Extraction
In this section we extract features from our dataset using a Vision Language Model (VLM).

This representation is computed once and then kept frozen offline. The goal is not to improve the VLM encoder itself, but to study and improve retrieval behavior on top of fixed CLIP embeddings.

In [ ]:
cache_dir = "/content/drive/MyDrive/datasets/clip_cache"
save_path = os.path.join(cache_dir, "embeddings.pt")

### Extracting features using a VLM
In this section we will be using the CLIP model to extract features from our dataset. We will be using the ViT-B/32 model, which is a smaller version of the original CLIP model. 

Since these embeddings are static, we can compute them offline and keep them frozen. This means that we don't have to worry about the computational cost of computing the embeddings during training, and we can also use a larger batch size for training our retriever model.

The result of this process will be a list of embeddings, where each embedding is of size 512 and corresponds to an image in our dataset. 

In [ ]:
if os.path.exists(save_path):
    print(f"Found cached embeddings at {save_path}")
    print("Loading cached embeddings...")
    data = torch.load(save_path)

    embeddings = data["embeddings"]
    print(f"Loaded embeddings with shape: {embeddings.shape}")
    labels = data["labels"]
    print(f"Loaded labels with shape: {labels.shape}")

else:
    print("Cache not found. Encoding dataset...")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    print("Loading CLIP model...")
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    print("Model loaded.")
    
    # Set the model to evaluation mode 
    model.eval()

    all_embeddings = []
    all_labels = []
    
    # Disable gradient computation for efficiency, since we are only doing inference
    print("Encoding images...")
    with torch.no_grad():
        for image, label in celeba:
            # Preprocess the image using the CLIP processor
            inputs = processor(images=image, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Get the image embeddings from the model
            feats = model.get_image_features(**inputs)
            if isinstance(feats, torch.Tensor):
                image_features = feats
            elif hasattr(feats, "pooler_output") and feats.pooler_output is not None:
                image_features = feats.pooler_output
            elif isinstance(feats, tuple) and len(feats) > 0:
                image_features = feats[0]
            else:
                raise TypeError(f"Unexpected image feature output type: {type(feats)}")

            # Normalize to unit sphere before storing so dot product == cosine similarity
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            # Move to CPU and store in list
            all_embeddings.append(image_features.cpu())
            all_labels.append(label)
    print("Encoding completed.")
    
    # Concatenate all embeddings and labels into tensors
    embeddings = torch.cat(all_embeddings, dim=0)
    labels = torch.stack(all_labels, dim=0)

    # Save the embeddings and labels to disk for future use
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save({
        "embeddings": embeddings,
        "labels": labels
    }, save_path)

    print("Saved embeddings to disk.")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Labels shape: {labels.shape}")

### Sanity check
Now that embeddings are available, let's run a quick sanity check to verify retrieval quality.

We will pick a source image, compare its CLIP embedding against all dataset embeddings, and inspect the nearest matches.

In [ ]:
source_idx = 10006
img, _ = celeba[source_idx]

plt.figure(figsize=(4, 4))
plt.imshow(img)

Now that we have our source image and its CLIP encoding, let's find the nearest embeddings in the dataset.

We normalize embeddings to unit vectors at extraction time, so similarity between any two embeddings reduces to a plain dot product.
For unit vectors, `dot(a, b) == cosine_similarity(a, b)`, so the metric is unchanged — but retrieval becomes a single matrix multiply instead of a per-pair loop.
We exclude the source image itself from the top results.

In [ ]:
if "embeddings" not in globals():
    raise RuntimeError(
        "Embeddings not found. Run the offline feature extraction cell above first."
    )

source_embedding = embeddings[source_idx]

# Dot product == cosine similarity for unit-norm embeddings (single matrix-vector multiply)
similarities = (embeddings @ source_embedding).tolist()

# Get the 5 highest-similarity matches, excluding the source image itself
nearest_indices = torch.argsort(torch.tensor(similarities), descending=True)[1:6].tolist()
nearest_similarities = [similarities[i] for i in nearest_indices]

print("Nearest indices:", nearest_indices)
print("Nearest cosine similarities:", nearest_similarities)

We have found the nearest embeddings! Let's visualize the nearest images to our source image and see if they are indeed similar.

In [ ]:
fig, axes = plt.subplots(ncols=5, figsize=(25, 5))

for i, image_idx in enumerate(nearest_indices):
    img, labels = celeba[image_idx]
    ax = axes[i]
    
    ax.set_title(f"Cosine sim: {nearest_similarities[i]:.4f}")
    
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Metrics

In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
):
    """
    Evaluate the retrieval performance for a single source image.

    Args:
    ----
        retrieved_indices: list of image IDs predicted by the model,
            ordered by similarity (descending).
        ground_truth_indices: list of valid target IDs from the benchmark JSON.
        k: the cutoff for top-K evaluation (e.g., 1, 5, 10).

    Return:
    ------
        A dictionary containing Recall@K and Precision@K.

    """
    # Isolate the top K predictions
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

In [ ]:
# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

# Evaluation

In [ ]:
import json

In [ ]:
annotations_path = Path("/content/drive/MyDrive/datasets/celeba_evaluation.json")

with open(annotations_path, "r") as f:
    annotations = json.load(f)

len(annotations)

In [ ]:
# `annotations` is a list of queries
# Each query is a dictionary with these keys:
# - `query`: the textual query itself
# - `ground_truth`: a dictionary of images
#
# Each element in `ground_truth` is structured as:
# {
#    idx: list[int]
# }
# `idx` is the source image that you have to use together with the `query`
# `list[int]` is the list of acceptable target images, i.e., the images
# that you should retrieve

print(annotations[0].keys())
print()
print("Query:", annotations[0]["query"])
print("Source images:", len(annotations[0]["ground_truth"].keys()))

Let's test the evaluation function. Let's simulate retrieving data for the first image / query.

In [ ]:
print("Nothing retrieved:\n", evaluate_retrieval([], annotations[0]["ground_truth"]["13"], 1))
print()
print("Retrieved only one wrong image:\n", evaluate_retrieval([0], annotations[0]["ground_truth"]["13"], 1))
print()
print("Retrieved 10 correct images:\n", evaluate_retrieval(annotations[0]["ground_truth"]["13"][:10], annotations[0]["ground_truth"]["13"], 1))
print()
# This returns 0 and 0 because top-k is set to 1, and the first 5 images
# (by highest similarity) are incorrect. We need to jump at least to top-6 here!
print("Retrieved 5 correct images and 5 wrong images:\n", evaluate_retrieval([0, 1, 2, 3, 4] + annotations[0]["ground_truth"]["13"][:5], annotations[0]["ground_truth"]["13"], 1))
print("Retrieved 5 correct images and 5 wrong images:\n", evaluate_retrieval([0, 1, 2, 3, 4] + annotations[0]["ground_truth"]["13"][:5], annotations[0]["ground_truth"]["13"], 6))

In [ ]:
# This is our source image
# Basically, the `key` in the provided JSON corresponds to the
# image index. Note, however, that the key is in string format,
# so remember to convert it with `int(key)`!
celeba[13][0]

In [ ]:
# This is one of the accepted matches
celeba[annotations[0]["ground_truth"]["13"][0]][0]

In [ ]:
# This is another one of the accepted matches
celeba[annotations[0]["ground_truth"]["13"][1]][0]

In [ ]:
# And yet another one of the accepted matches
celeba[annotations[0]["ground_truth"]["13"][2]][0]

As you can see, all these images are pretty similar to the first one (the "source").
This is exactly what we want.
We constructed the ground truth by taking images that match the given query, with a Hamming distance of at most 2 from the source (i.e., having a perfect match for the query is impossible, so we allow some small variability on other attributes, too).